In [1]:
import duckdb as ddb
import pandas as pd
import plotly.graph_objects as go

con = ddb.connect(database=":memory:")

con.sql("""
CREATE TABLE program_beslut AS
    FROM 'data/resultat-ansokningsomgang-2020-2024-beslut.csv';
CREATE TABLE program_kommun AS
    FROM 'data/resultat-ansokningsomgang-2020-2024-diarie_kommun.csv';

CREATE TABLE kurser_beslut AS
    FROM 'data/resultat-for-kurser-inom-yh-2024-beslut.csv';
CREATE TABLE kurser_kommun AS
    FROM 'data/resultat-for-kurser-inom-yh-2024-diarie_kommun.csv';

CREATE TABLE kommun_lan AS
    FROM 'data/kommunlankod-2025.csv';       
""")

In [2]:
def create_rel_program_alla_kommuner(
    con,
    years: list[int] | int | None = None,
    *,
    distinct=False,
) -> ddb.DuckDBPyRelation:
    if not years:
        years = None
    elif isinstance(years, int):
        years = [years]

    query = """
        with program_alla_kommuner as (

        select
            "Utbildningsområde",
            "Utbildningsnamn",
            "Län",
            "Kommun",
            "Antal kommuner",
            "Flera kommuner",
            "YH-poäng",
            "Studieform",
            "Studietakt %" as "Studietakt %",
            "Utbildningsanordnare",
            "Huvudmannatyp",
            "Sökta utbildningsomgångar",
            "Beviljade utbildningsomgångar",
            "Sökta platser totalt",
            "Beviljade platser totalt",
            "Sökta platser per utbildningsomgång",
            "Ansökningsomgång",
            "Diarienummer",
            "Beslut"
        from program_beslut pb
        where
            "Flera kommuner" is not TRUE
            and ( $year IS NULL OR pb."Ansökningsomgång" = ANY($year) )
        
        union all

        select
            pb."Utbildningsområde",
            pb."Utbildningsnamn",
            pk."Län",                -- kurser_kommun
            pk."Kommun",             -- kurser_kommun
            pb."Antal kommuner",
            pb."Flera kommuner",
            pb."YH-poäng",
            pb."Studieform",
            pb."Studietakt %" as "Studietakt %",
            pb."Utbildningsanordnare",
            pb."Huvudmannatyp",
            pb."Sökta utbildningsomgångar",
            pb."Beviljade utbildningsomgångar",
            pb."Sökta platser totalt",
            pb."Beviljade platser totalt",
            pb."Sökta platser per utbildningsomgång",
            pb."Ansökningsomgång",
            pb."Diarienummer",
            pb."Beslut"
        from program_beslut pb
        join program_kommun pk
            on pk."Diarienummer" = pb."Diarienummer"
        where
            pb."Flera kommuner" is TRUE
            and ( $year IS NULL OR pb."Ansökningsomgång" = ANY($year) )

        )
        select distinct on ("Diarienummer", "Kommun")
            *
        from program_alla_kommuner
        order by "Diarienummer";"""

    rel = con.sql(query, params={"year": years})
    return rel

In [3]:
rel_program = create_rel_program_alla_kommuner(con, distinct=True)

rel_program.describe()

┌─────────┬───────────────────┬────────────────────────────────────────────────────────────────────┬──────────────┬──────────────┬────────────────────┬────────────────┬───────────────────┬────────────┬───────────────────┬─────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┬────────────────────┬───────────────┬─────────┐
│  aggr   │ Utbildningsområde │                          Utbildningsnamn                           │     Län      │    Kommun    │   Antal kommuner   │ Flera kommuner │     YH-poäng      │ Studieform │   Studietakt %    │      Utbildningsanordnare       │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │  Ansökningsomgång  │ Diarienummer  │ Beslut  │
│ varchar │      varchar      │                              varch

In [4]:
def create_rel_program_omrade(con):
    rel = con.sql(
        """
        with utbildningsanordnare as (
            select
                Ansökningsomgång,
                Utbildningsområde,
                Utbildningsanordnare,
                count(distinct Diarienummer) as diarie_anordnare
            from 
                rel_program
            group by
                Ansökningsomgång,
                Utbildningsområde,
                Utbildningsanordnare,           
        )

        select
            rp.Ansökningsomgång,
            rp.Utbildningsområde,
            count(distinct rp.Diarienummer) as diarie_total,
            count(distinct case when Beslut = TRUE then rp.Diarienummer end) as diarie_approved,
            round(avg(ua.diarie_anordnare), 1) as diarie_anordnare_mean,
            round(sum(rp."Sökta platser totalt"), 0)::integer as seats_total,
            round(sum(rp."Beviljade platser totalt"), 0)::integer as seats_approved_total,
            round(100.00 * sum(rp."Beviljade platser totalt") / NULLIF(sum(rp."Sökta platser totalt"), 0), 1) as seats_approved_total_pct,
            round(avg(rp."Sökta platser totalt"), 0)::integer as seats_mean,
            round(avg(rp."Beviljade platser totalt"), 0)::integer as seats_approved_mean,
            round(100.00 * avg(rp."Beviljade platser totalt") / NULLIF(avg(rp."Sökta platser totalt"), 0), 1) as seats_approved_mean_pct,
            count(distinct case when Studieform = 'Distans' then rp.Diarienummer end) as distans_total,
        from
            rel_program rp
        left join
            utbildningsanordnare as ua
            on rp.Utbildningsområde = ua.Utbildningsområde
            and rp.Ansökningsomgång = ua.Ansökningsomgång
        group by
            rp.Ansökningsomgång, rp.Utbildningsområde
        order by rp.Ansökningsomgång, diarie_total desc
        """
    )
    return rel


rel_program_omrade = create_rel_program_omrade(con)

rel_program_omrade.describe()

┌─────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────────┬───────────────────┬──────────────────────┬──────────────────────────┬───────────────────┬─────────────────────┬─────────────────────────┬────────────────────┐
│  aggr   │  Ansökningsomgång  │ Utbildningsområde │   diarie_total    │  diarie_approved  │ diarie_anordnare_mean │    seats_total    │ seats_approved_total │ seats_approved_total_pct │    seats_mean     │ seats_approved_mean │ seats_approved_mean_pct │   distans_total    │
│ varchar │       double       │      varchar      │      double       │      double       │        double         │      double       │        double        │          double          │      double       │       double        │         double          │       double       │
├─────────┼────────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────────┼───────────────────┼──────────────────────┼─────────────

In [5]:
df = rel_program_omrade.df()

df.describe()

,Ansökningsomgång,diarie_total,diarie_approved,diarie_anordnare_mean,seats_total,seats_approved_total,seats_approved_total_pct,seats_mean,seats_approved_mean,seats_approved_mean_pct,distans_total
count,75.000000,75.000000,75.000000,75.000000,7.500000e+01,7.500000e+01,75.000000,75.000000,75.000000,75.000000,75.000000
mean,2022.000000,86.093333,28.680000,1.965333,7.300633e+05,1.796107e+05,28.333333,99.733333,29.840000,28.333333,40.106667
std,1.423737,95.655516,32.566008,0.838998,1.076020e+06,2.653331e+05,17.725233,23.899188,29.867745,17.725233,45.193151
min,2020.000000,1.000000,1.000000,1.000000,1.750000e+02,9.000000e+01,4.000000,58.000000,3.000000,4.000000,0.000000
25%,2021.000000,12.000000,3.000000,1.300000,1.316000e+04,2.690000e+03,16.900000,81.000000,16.500000,16.900000,4.500000
50%,2022.000000,38.000000,13.000000,1.800000,8.328600e+04,2.460600e+04,25.600000,97.000000,23.000000,25.600000,15.000000
75%,2023.000000,168.000000,54.000000,2.300000,1.397211e+06,3.707010e+05,34.550000,112.000000,33.000000,34.550000,90.500000
max,2024.000000,313.000000,125.000000,4.200000,3.870856e+06,1.159104e+06,100.000000,175.000000,175.000000,100.000000,132.000000


In [6]:
import plotly.graph_objects as go

# Filter the dataframe for Data/IT
data_it_df = df[df['Utbildningsområde'] == 'Data/IT']

# Create the figure
fig = go.Figure()

# Add traces for all numeric columns
fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['diarie_total'],
    name='Total Applications',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['diarie_approved'],
    name='Approved Applications',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['diarie_anordnare_mean'],
    name='Mean Applications per Organizer',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_total'],
    name='Total Seats',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_approved_total'],
    name='Total Approved Seats',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_approved_total_pct'],
    name='Approved Seats %',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_mean'],
    name='Mean Seats',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_approved_mean'],
    name='Mean Approved Seats',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['seats_approved_mean_pct'],
    name='Mean Approved Seats %',
    mode='lines+markers'
))

fig.add_trace(go.Scatter(
    x=data_it_df['Ansökningsomgång'],
    y=data_it_df['distans_total'],
    name='Total Distance Learning',
    mode='lines+markers'
))

# Update layout
fig.update_layout(
    title='Data/IT Education Area Metrics Over Time',
    xaxis_title='Application Round (Year)',
    yaxis_title='Values',
    showlegend=True,
    hovermode='x unified',
    height=800,  # Make the plot taller for better visibility
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.05
    )
)

# Show the plot
fig.show()


In [21]:
df = con.sql(
    """
  select
    Ansökningsomgång,
    Utbildningsområde,
    sum("Sökta platser totalt") as seats_total,
    sum("Beviljade platser totalt") as seats_approved_total
from
    rel_program
-- where Ansökningsomgång = 2024
group by
    Ansökningsomgång, Utbildningsområde
order by
    seats_total desc


    """
).df()

df

,Ansökningsomgång,Utbildningsområde,seats_total,seats_approved_total
0,2023,Data/IT,49054.0,9904.0
1,2024,Data/IT,46543.0,5788.0
2,2024,"Ekonomi, administration och försäljning",43987.0,6175.0
3,2020,"Ekonomi, administration och försäljning",36961.0,6403.0
4,2023,"Ekonomi, administration och försäljning",34644.0,5802.0
...,...,...,...,...
70,2024,Övrigt,350.0,350.0
71,2021,Övrigt,231.0,105.0
72,2022,Övrigt,175.0,140.0
73,2020,Övrigt,175.0,175.0


In [22]:
import plotly.graph_objects as go

# Get unique Utbildningsområden
områden = df['Utbildningsområde'].unique()

# Create the figure
fig = go.Figure()

# Add a line for each Utbildningsområde
for område in områden:
    område_df = df[df['Utbildningsområde'] == område].sort_values(by='Ansökningsomgång')
    fig.add_trace(go.Scatter(
        x=område_df['Ansökningsomgång'],  # Years on x-axis
        y=område_df['seats_approved_total'],
        mode='lines+markers',  # Lines with markers
        name=område  # Legend label
    ))

# Customize the layout
fig.update_layout(
    title='Approved Seats Over Time by Utbildningsområde',
    xaxis_title='Application Round (Year)',
    yaxis_title='Approved Seats Total',
    xaxis=dict(
        tickmode='array',  # Ensure all years are shown on x-axis
        tickvals=df['Ansökningsomgång'].unique(),
        tickangle=-45  # Rotate for readability
    ),
    legend=dict(
        orientation='h',  # Horizontal legend
        yanchor='bottom',
        y=1.05,
        xanchor='center',
        x=0.5
    )
)

fig.show()
